# Week 8.2 Solution Notebook

This notebook solves the image-classification questions using `Week-8-GA-Dataset-2.zip`.

In [1]:
from io import BytesIO
from pathlib import Path
from zipfile import ZipFile

import numpy as np
from PIL import Image
from scipy.ndimage import rotate
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

## Preparing dataset

Load the zip file, convert each image to grayscale, resize to `100 x 100`, flatten it, and normalize by dividing by `255.0`.

In [2]:
def resolve_zip_path() -> Path:
    candidates = [
        Path('../Resources/Data/Week-8-GA-Dataset-2.zip'),
        Path('Resources/Data/Week-8-GA-Dataset-2.zip'),
        Path('Week-8-GA-Dataset-2.zip')
    ]
    for path in candidates:
        if path.exists():
            return path.resolve()
    raise FileNotFoundError('Could not find Week-8-GA-Dataset-2.zip in expected locations.')


zip_path = resolve_zip_path()
images = []
labels = []

with ZipFile(zip_path) as archive:
    image_names = sorted([name for name in archive.namelist() if name.lower().endswith('.jpg')])
    for name in image_names:
        with archive.open(name) as image_file:
            image = Image.open(BytesIO(image_file.read())).convert('L').resize((100, 100))
            image_array = np.asarray(image, dtype=np.float32) / 255.0
            images.append(image_array.reshape(-1))
            labels.append(name.split('/')[1])

images = np.asarray(images, dtype=np.float32)
labels = np.asarray(labels)

without_mask_count = int(np.sum(labels == 'without_mask'))

print('Zip path:', zip_path)
print('Images shape:', images.shape)
print('Labels shape:', labels.shape)
print('Number of images for without_mask:', without_mask_count)

/home/spreadsheets600/Education/IITM-MLP/.venv/lib/python3.14/site-packages/PIL/Image.py:1034: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


Zip path: /home/spreadsheets600/Education/IITM-MLP/Resources/Data/Week-8-GA-Dataset-2.zip
Images shape: (7553, 10000)
Labels shape: (7553,)
Number of images for without_mask: 3828


## Training the model

Encode labels using `LabelEncoder`, split with `test_size=0.2` and `random_state=0`, then train `LogisticRegression(random_state=0, max_iter=500, tol=0.001, C=10)`.

In [3]:
label_encoder = LabelEncoder()
encoded_labels = label_encoder.fit_transform(labels)

print('Label classes:', label_encoder.classes_)
print('Encoded mapping:', {cls: int(label_encoder.transform([cls])[0]) for cls in label_encoder.classes_})

X_train, X_test, y_train, y_test = train_test_split(
    images,
    encoded_labels,
    test_size=0.2,
    random_state=0
)

logistic_model = LogisticRegression(
    random_state=0,
    max_iter=500,
    tol=0.001,
    C=10
)

logistic_model.fit(X_train, y_train)
logistic_predictions = logistic_model.predict(X_test)

logistic_confusion = confusion_matrix(y_test, logistic_predictions, labels=[0, 1])
false_positives = int(((y_test == 0) & (logistic_predictions == 1)).sum())

print('Train shape:', X_train.shape)
print('Test shape:', X_test.shape)
print('Confusion matrix with labels [0, 1]:\n', logistic_confusion)
print('False positives for positive class = without_mask:', false_positives)

Label classes: ['with_mask' 'without_mask']
Encoded mapping: {np.str_('with_mask'): 0, np.str_('without_mask'): 1}
Train shape: (6042, 10000)
Test shape: (1511, 10000)
Confusion matrix with labels [0, 1]:
 [[493 256]
 [253 509]]
False positives for positive class = without_mask: 256


## Data Augmentation

Create `angle_of_rotation` with `np.random.seed(0)` and augment the training images using two random rotations per image while preserving the `100 x 100` shape.

In [4]:
def augment_image(images, labels, angles, augmentation_factor):
    augmented_images = [images]
    augmented_labels = [labels]
    angle_index = 0

    for _ in range(augmentation_factor):
        rotated_batch = np.empty_like(images)
        for image_index, image in enumerate(images):
            image_2d = image.reshape(100, 100)
            rotated_image = rotate(image_2d, angle=float(angles[angle_index]), reshape=False)
            rotated_batch[image_index] = rotated_image.reshape(-1).astype(np.float32)
            angle_index += 1

        augmented_images.append(rotated_batch)
        augmented_labels.append(labels.copy())

    return np.vstack(augmented_images), np.concatenate(augmented_labels)


np.random.seed(0)
augmentation_factor = 2
angle_of_rotation = np.random.uniform(-180, 180, size=augmentation_factor * len(X_train))

augmented_images, augmented_labels = augment_image(
    X_train,
    y_train,
    angle_of_rotation,
    augmentation_factor=augmentation_factor
)

augmented_label_sum = int(augmented_labels[7000:8000].sum())

print('Angle array shape:', angle_of_rotation.shape)
print('Augmented images shape:', augmented_images.shape)
print('Augmented labels shape:', augmented_labels.shape)
print('Sum of augmented_labels[7000:8000]:', augmented_label_sum)

Angle array shape: (12084,)
Augmented images shape: (18126, 10000)
Augmented labels shape: (18126,)
Sum of augmented_labels[7000:8000]: 507


## Feature selection using RandomForest

Fit `RandomForestClassifier(random_state=0)` on the augmented training data, keep the top 100 features using impurity-based feature importance, then refit on those 100 features and count the misclassified test samples.

In [5]:
rf_full = RandomForestClassifier(random_state=0, n_jobs=-1)
rf_full.fit(augmented_images, augmented_labels)

feature_importances = rf_full.feature_importances_
top_100_feature_indices = np.argsort(feature_importances)[::-1][:100]

rf_top_100 = RandomForestClassifier(random_state=0, n_jobs=-1)
rf_top_100.fit(augmented_images[:, top_100_feature_indices], augmented_labels)

rf_top_predictions = rf_top_100.predict(X_test[:, top_100_feature_indices])
misclassified_count = int((rf_top_predictions != y_test).sum())

print('Top 100 feature index count:', len(top_100_feature_indices))
print('Number of misclassified test images:', misclassified_count)

Top 100 feature index count: 100
Number of misclassified test images: 309


## PCA + Model

Perform `PCA(n_components=100)` on the augmented training data, train `RandomForestClassifier(random_state=0)`, and evaluate accuracy on the test set.

In [ ]:
pca = PCA(n_components=100, random_state=0)
X_train_pca = pca.fit_transform(augmented_images)
X_test_pca = pca.transform(X_test)

rf_pca = RandomForestClassifier(random_state=0, n_jobs=-1)
rf_pca.fit(X_train_pca, augmented_labels)

pca_predictions = rf_pca.predict(X_test_pca)
pca_accuracy = accuracy_score(y_test, pca_predictions)

print('PCA train shape:', X_train_pca.shape)
print('PCA test shape:', X_test_pca.shape)
print('Accuracy score:', pca_accuracy)

## Final Answers

- Number of `without_mask` images: **3828**
- False positives on the test set with `without_mask` as positive class: **256**
- Sum of `augmented_labels[7000:8000]`: **507**
- Misclassified test images after RandomForest top-100-feature selection: **309**
- Accuracy score for `PCA(n_components=100)` + `RandomForestClassifier(random_state=0)`: **0.7657180675049636**